# Attention U-Net: Class-Imbalance-Aware Training Strategies
### Pancreas segmentation, extension of Oktay et al. (MIDL 2018)

This notebook contains the full set of experiments used in the final report:
eight model/loss combinations trained for 15 epochs each, the seven most relevant
of them extended to 50 epochs, and clean evaluation on a held-out test set that
never influences any training or model-selection decision.



## 1. Setup
Mounts Drive, pulls the repository, installs dependencies, and confirms the environment.

In [ ]:
import os, json, subprocess

from google.colab import drive
drive.mount('/content/drive')

REPO_PATH = '/content/Attention-U-net-DL_Project'
REPO_URL = 'https://github.com/MAR7142/Attention-U-net-DL_Project.git'

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
%cd {REPO_PATH}
!git pull

!pip install -q git+https://github.com/ozan-oktay/torchsample.git
!pip install -q dominate
!pip install -q visdom

print("=== num_workers currently set to: ===")
!grep -n "num_workers=" train_segmentation.py

print("\n=== Available criteria and extensions in this repo ===")
!grep -n "criterion ==" models/utils.py
!grep -n "use_residual" models/layers/grid_attention_layer.py | head -3


## 2. Dataset

The dataset is the public TCIA Pancreas-CT collection (CT-82 benchmark), the same one
used in the original paper's second experiment (80 usable scans after TCIA removed
2 duplicate cases from the original 82).

### 2.1 Preprocessing

This step converts the raw DICOM series to NIfTI volumes and builds a 64/16
(train/test) patient-level split from the full 80-scan collection. Patient/label
pairing uses the DICOM `PatientID` tag rather than the download folder name, since
the folder name is a Series UID and does not reliably identify the patient. The
output, `pancreas_data_prepared`, is the source for the smaller working subset
built in Section 2.2.

In [ ]:
PREPARED_ROOT = '/content/drive/MyDrive/pancreas_data_prepared'

if os.path.exists(PREPARED_ROOT):
    print("Prepared dataset found:", PREPARED_ROOT)
else:
    print("Preprocessing raw DICOM series into", PREPARED_ROOT)
    print("Reads PatientID from DICOM tag (0010,0020), orders slices by true spatial")
    print("position, and splits 64 train / 16 test (seed=42).")
    !python preprocess_dicom.py


### 2.2 Working subset (24 train / 6 validation / 6 test)

All experiments use a 24-image training subset, drawn from the 64-image prepared
training set, given the compute available for this project. Validation and test
are disjoint 6-image sets, each drawn from the 16-image prepared test set, so no
image appears in more than one split.

In [ ]:
import shutil

SRC_ROOT = '/content/drive/MyDrive/pancreas_data_prepared'
DST_ROOT = '/content/drive/MyDrive/pancreas_data_medium'

def build_medium_subset(src_root=SRC_ROOT, dst_root=DST_ROOT, n_train=24, n_val=6, n_test=6):
    if os.path.exists(dst_root):
        print("Subset already exists, skipping rebuild:", dst_root)
        return

    # --- Train: first n_train images from the prepared train split ---
    src_img_dir = os.path.join(src_root, 'train', 'image')
    src_lbl_dir = os.path.join(src_root, 'train', 'label')
    train_images = sorted(os.listdir(src_img_dir))[:n_train]
    train_labels = sorted(os.listdir(src_lbl_dir))[:n_train]
    for imgs, lbls, split in [(train_images, train_labels, 'train')]:
        img_dst = os.path.join(dst_root, split, 'image')
        lbl_dst = os.path.join(dst_root, split, 'label')
        os.makedirs(img_dst, exist_ok=True)
        os.makedirs(lbl_dst, exist_ok=True)
        for f in imgs:
            shutil.copy2(os.path.join(src_img_dir, f), os.path.join(img_dst, f))
        for f in lbls:
            shutil.copy2(os.path.join(src_lbl_dir, f), os.path.join(lbl_dst, f))

    # --- Validation and test: DISJOINT slices of the prepared test split ---
    src_img_dir = os.path.join(src_root, 'test', 'image')
    src_lbl_dir = os.path.join(src_root, 'test', 'label')
    all_images = sorted(os.listdir(src_img_dir))
    all_labels = sorted(os.listdir(src_lbl_dir))

    val_images, val_labels = all_images[:n_val], all_labels[:n_val]
    test_images, test_labels = all_images[n_val:n_val + n_test], all_labels[n_val:n_val + n_test]

    for imgs, lbls, split in [(val_images, val_labels, 'validation'), (test_images, test_labels, 'test')]:
        img_dst = os.path.join(dst_root, split, 'image')
        lbl_dst = os.path.join(dst_root, split, 'label')
        os.makedirs(img_dst, exist_ok=True)
        os.makedirs(lbl_dst, exist_ok=True)
        for f in imgs:
            shutil.copy2(os.path.join(src_img_dir, f), os.path.join(img_dst, f))
        for f in lbls:
            shutil.copy2(os.path.join(src_lbl_dir, f), os.path.join(lbl_dst, f))

    overlap = set(val_images) & set(test_images)
    print("Train:", train_images)
    print("Validation:", val_images)
    print("Test:", test_images)
    print("Validation/test overlap (must be empty):", overlap)
    assert not overlap, "Validation and test overlap! Do not proceed."

build_medium_subset()


## 3. Training helper

A single reusable function builds the config JSON and launches `train_segmentation.py`,
so each experiment below is one short function call instead of a rewritten config block.

In [ ]:
def run_training(arch='baseline', criterion='dice_loss', checkpoints_dir=None,
                  n_epochs=15, class_weight=None, focal_gamma=None,
                  use_residual=False, continue_train=False, which_epoch=-1,
                  data_path='/content/drive/MyDrive/pancreas_data_medium'):
    """
    arch: 'baseline' (plain U-Net) or 'attention' (Attention U-Net)
    criterion: 'dice_loss', 'weighted_cross_entropy', or 'focal_loss'
    checkpoints_dir: Drive folder for this run's checkpoints (own folder per experiment)
    """
    config_path = ('configs/config_unet_ct_dsv.json' if arch == 'baseline'
                    else 'configs/config_unet_ct_multi_att_dsv.json')

    with open(config_path) as f:
        config = json.load(f)

    config['data_path']['acdc_sax'] = data_path
    config['model']['output_nc'] = 2
    config['training']['n_epochs'] = n_epochs
    config['training']['preloadData'] = False
    config['training']['save_epoch_freq'] = 1
    config['model']['checkpoints_dir'] = checkpoints_dir
    config['model']['continue_train'] = continue_train
    config['model']['which_epoch'] = which_epoch
    config['model']['criterion'] = criterion
    if class_weight is not None:
        config['model']['class_weight'] = class_weight
    if focal_gamma is not None:
        config['model']['focal_gamma'] = focal_gamma
    if arch == 'attention':
        config['model']['use_residual_attention'] = use_residual

    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)

    print(f"--- Running: arch={arch}, criterion={criterion}, residual={use_residual}, "
          f"epochs={n_epochs}, resume_from={which_epoch if continue_train else 'scratch'} ---")
    print(json.dumps(config['model'], indent=2))

    !python train_segmentation.py --config {config_path}


## 4. Matched comparison: 8 configurations, 15 epochs each

Every combination of architecture (baseline / attention), loss function
(Dice / Weighted Cross-Entropy / Focal Loss), and the residual-attention
variant, all trained for the same 15 epochs for a fair comparison.


In [ ]:
# 4.1 Baseline U-Net + Dice (original paper's loss, no changes)
run_training(arch='baseline', criterion='dice_loss',
             checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_dice_15ep',
             n_epochs=15)


In [ ]:
# 4.2 Attention U-Net + Dice (original paper's full model, unmodified)
run_training(arch='attention', criterion='dice_loss',
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_dice_15ep',
             n_epochs=15)


In [ ]:
# 4.3 Baseline U-Net + Weighted Cross-Entropy (alpha=30)
run_training(arch='baseline', criterion='weighted_cross_entropy', class_weight=30.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_weightedce_15ep',
             n_epochs=15)


In [ ]:
# 4.4 Attention U-Net + Weighted Cross-Entropy (alpha=30) - primary extension
run_training(arch='attention', criterion='weighted_cross_entropy', class_weight=30.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_weightedce_15ep',
             n_epochs=15)


In [ ]:
# 4.5 Attention U-Net + Residual attention gate + Weighted Cross-Entropy (alpha=30)
run_training(arch='attention', criterion='weighted_cross_entropy', class_weight=30.0,
             use_residual=True,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_weightedce_15ep',
             n_epochs=15)


In [ ]:
# 4.6 Attention U-Net + Residual attention gate + Dice (isolates the residual effect alone)
run_training(arch='attention', criterion='dice_loss', use_residual=True,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_dice_15ep',
             n_epochs=15)


In [ ]:
# 4.7 Baseline U-Net + Focal Loss (gamma=2, alpha=30)
run_training(arch='baseline', criterion='focal_loss', class_weight=30.0, focal_gamma=2.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_focal_15ep',
             n_epochs=15)


In [ ]:
# 4.8 Attention U-Net + Focal Loss (gamma=2, alpha=30)
run_training(arch='attention', criterion='focal_loss', class_weight=30.0, focal_gamma=2.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_focal_15ep',
             n_epochs=15)


## 5. Extended training: the most relevant configurations to 50 epochs

Resumes from each run's epoch-14 checkpoint (`continue_train=True, which_epoch=14`)
and continues to epoch 49. Baseline + Dice is not extended, since it is only used
as the reference point in the 15-epoch table.

Note: because the training loop's epoch range includes `which_epoch` itself, epoch 14
is retrained once more (overwriting its own checkpoint) before continuing to 49. This
is expected and does not affect the final results.

In [ ]:
# 5.1 Attention U-Net + Dice, extended
run_training(arch='attention', criterion='dice_loss',
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_dice_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.2 Attention U-Net + Residual + Dice, extended
run_training(arch='attention', criterion='dice_loss', use_residual=True,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_dice_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.3 Attention U-Net + Weighted Cross-Entropy, extended
run_training(arch='attention', criterion='weighted_cross_entropy', class_weight=30.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_weightedce_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.4 Attention U-Net + Residual + Weighted Cross-Entropy, extended
run_training(arch='attention', criterion='weighted_cross_entropy', class_weight=30.0,
             use_residual=True,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_weightedce_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.5 Attention U-Net + Focal Loss, extended
run_training(arch='attention', criterion='focal_loss', class_weight=30.0, focal_gamma=2.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_focal_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.6 Baseline U-Net + Weighted Cross-Entropy, extended
run_training(arch='baseline', criterion='weighted_cross_entropy', class_weight=30.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_weightedce_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


In [ ]:
# 5.7 Baseline U-Net + Focal Loss, extended
run_training(arch='baseline', criterion='focal_loss', class_weight=30.0, focal_gamma=2.0,
             checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_focal_15ep',
             n_epochs=50, continue_train=True, which_epoch=14)


## 6. Clean evaluation on the held-out test set

`evaluate_on_test.py` loads a saved checkpoint and evaluates it
on the `test` split, reusing the same metric code
already verified during training. For the 15-epoch runs, epoch 14 is evaluated
directly. For the 50-epoch runs, the epoch is first chosen by the **best pancreas
Dice on validation**, read from that run's
`loss_log.txt`, and then evaluated on test.

In [ ]:
import re

def find_best_val_epoch(checkpoints_dir, arch='attention', exclude_epoch=None):
    """Parses loss_log.txt and returns the epoch with the highest validation
    Class_1 (pancreas) Dice. Optionally excludes one epoch (e.g. epoch 0, if it
    reflects a near-random initial state rather than a trained model)."""
    exp_name = 'experiment_unet_ct_dsv_big' if arch == 'baseline' else 'experiment_unet_ct_multi_att_dsv'
    log_path = os.path.join(checkpoints_dir, exp_name, 'loss_log.txt')
    best_epoch, best_val = None, -1.0
    with open(log_path) as f:
        for line in f:
            m = re.match(r"\(epoch: (\d+), split: validation\).*Class_1: ([\d.]+)", line)
            if m:
                epoch, val_c1 = int(m.group(1)), float(m.group(2))
                if exclude_epoch is not None and epoch == exclude_epoch:
                    continue
                if val_c1 > best_val:
                    best_val, best_epoch = val_c1, epoch
    return best_epoch, best_val


def evaluate_checkpoint(arch, checkpoints_dir, which_epoch, criterion,
                         class_weight=None, focal_gamma=None, use_residual=False):
    config_path = ('configs/config_unet_ct_dsv.json' if arch == 'baseline'
                    else 'configs/config_unet_ct_multi_att_dsv.json')
    with open(config_path) as f:
        config = json.load(f)
    config['data_path']['acdc_sax'] = '/content/drive/MyDrive/pancreas_data_medium'
    config['model']['output_nc'] = 2
    config['model']['checkpoints_dir'] = checkpoints_dir
    config['model']['criterion'] = criterion
    if class_weight is not None:
        config['model']['class_weight'] = class_weight
    if focal_gamma is not None:
        config['model']['focal_gamma'] = focal_gamma
    if arch == 'attention':
        config['model']['use_residual_attention'] = use_residual
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)

    !python evaluate_on_test.py -c {config_path} -e {which_epoch}


In [ ]:
# Evaluates every 15-epoch checkpoint (epoch 14) on the clean, held-out test set.
# The 50-epoch checkpoints are evaluated separately below, using find_best_val_epoch()
# to select each run's epoch by validation score first.

runs_15ep = [
    dict(arch='baseline',  checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_dice_15ep',
         criterion='dice_loss'),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_dice_15ep',
         criterion='dice_loss'),
    dict(arch='baseline',  checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0, use_residual=True),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_dice_15ep',
         criterion='dice_loss', use_residual=True),
    dict(arch='baseline',  checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_focal_15ep',
         criterion='focal_loss', class_weight=30.0, focal_gamma=2.0),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_focal_15ep',
         criterion='focal_loss', class_weight=30.0, focal_gamma=2.0),
]

for run in runs_15ep:
    evaluate_checkpoint(which_epoch=14, **run)


In [ ]:
# 50-epoch checkpoints: select each run's best epoch by validation score first, then evaluate.
# Attention + Residual + Dice is a special case: its true peak is epoch 0 (near-random
# initialization), which is not a meaningfully trained model, so epoch 0 is excluded and
# the best epoch after training actually begins is reported instead.

runs_50ep = [
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_dice_15ep',
         criterion='dice_loss'),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_dice_15ep',
         criterion='dice_loss', use_residual=True, exclude_epoch=0),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_residual_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0, use_residual=True),
    dict(arch='attention', checkpoints_dir='/content/drive/MyDrive/checkpoints/attention_focal_15ep',
         criterion='focal_loss', class_weight=30.0, focal_gamma=2.0),
    dict(arch='baseline',  checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_weightedce_15ep',
         criterion='weighted_cross_entropy', class_weight=30.0),
    dict(arch='baseline',  checkpoints_dir='/content/drive/MyDrive/checkpoints/baseline_focal_15ep',
         criterion='focal_loss', class_weight=30.0, focal_gamma=2.0),
]

for run in runs_50ep:
    exclude = run.pop('exclude_epoch', None)
    best_epoch, best_val = find_best_val_epoch(run['checkpoints_dir'], arch=run['arch'], exclude_epoch=exclude)
    print(f"Best epoch on validation for {run['checkpoints_dir']}: {best_epoch} (val Class_1 = {best_val:.3f})")
    evaluate_checkpoint(which_epoch=best_epoch, **run)


## 7. Class-imbalance measurement (for reference)

This is the diagnostic used once to measure how much of each training patch is
actually pancreas, which motivated the Weighted Cross-Entropy and Focal Loss
extensions in Section 2.1 of the report. It is not required to reproduce the
main results and can be skipped.

In [ ]:
import torch
import numpy as np
from dataio.loader import get_dataset, get_dataset_path
from dataio.transformation import get_dataset_transformation
from utils.util import json_file_to_pyobj
from torch.utils.data import DataLoader


config_path = 'configs/config_unet_ct_dsv.json'
with open(config_path) as f:
    _cfg = json.load(f)
_cfg['data_path']['acdc_sax'] = '/content/drive/MyDrive/pancreas_data_medium'
with open(config_path, 'w') as f:
    json.dump(_cfg, f, indent=4)

json_opts = json_file_to_pyobj(config_path)
ds_class = get_dataset(json_opts.training.arch_type)
ds_path = get_dataset_path(json_opts.training.arch_type, json_opts.data_path)
ds_transform = get_dataset_transformation(json_opts.training.arch_type, opts=json_opts.augmentation)
train_dataset = ds_class(ds_path, split='train', transform=ds_transform['train'])
loader = DataLoader(dataset=train_dataset, batch_size=2, shuffle=True, num_workers=0)

fractions = []
for i, (images, labels) in enumerate(loader):
    frac = (labels == 1).float().mean().item()
    fractions.append(frac)
    print(f"batch {i}: foreground fraction = {frac:.6f}")
    if i >= 10:
        break

print(f"\nRange observed: {min(fractions):.4%} to {max(fractions):.4%}")
